In [150]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

In [151]:
Path_E = r"F:\graduation project\data\Code\final\ecg 60s 10min\features\ECG125.csv"
ECG = pd.read_csv(Path_E, sep=',')

Path_A = r"F:\graduation project\data\Code\final\ACC 60s 10min\features\ACC125.csv"
ACC = pd.read_csv(Path_A, sep=',')

In [152]:
print(ECG.columns)

Index(['subject', 'run', 'sex', 'start_time', 'end_time', 'eventtype', 'label',
       'signal', 'times', 'R_peaks', 'RR_intervals_ms', 'HR_series', 'status',
       'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms',
       'RMSSD_ms', 'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg',
       'ptp_ecg', 'total_power_ecg', 'bp_0_5_3_ecg', 'n_beats'],
      dtype='object')


In [153]:
def convert(df, label_col='label'):

    df = df.copy()
    df.loc[df[label_col] == 'postictal', label_col] = 'normal'
    return df
ECG = convert(ECG)
ACC = convert(ACC)

In [ ]:
cols = ["acc_magnitude", "activity_score", "rms_mag", "std_mag"]

for c in cols:
    ACC[c] = pd.to_numeric(ACC[c], errors="coerce")

ACC[cols] = ACC[cols].fillna(0)

ACC["acc_score"] = (
    ACC["acc_magnitude"] +
    ACC["activity_score"] +
    ACC["rms_mag"] +
    ACC["std_mag"]
) / 4

ACC["acc_score"] = (
    ACC["acc_score"] - ACC["acc_score"].min()
) / (ACC["acc_score"].max() - ACC["acc_score"].min() + 1e-8)

In [155]:
Label_counts = ACC['label'].value_counts().reset_index()
Label_counts.columns = ['Label', 'Count']
Label_counts

,Label,Count
0,normal,540
1,preictal,112
2,ictal,86


In [156]:
Label_counts = ECG['label'].value_counts().reset_index()
Label_counts.columns = ['Label', 'Count']
Label_counts

,Label,Count
0,normal,540
1,preictal,112
2,ictal,86


In [157]:
import joblib

model_path = r"F:\graduation project\data\Code\final\ecg 60s 10min\patient_models100_csv\patient_125.0_model.pkl"
model = joblib.load(model_path)

In [158]:
ECG.columns

Index(['subject', 'run', 'sex', 'start_time', 'end_time', 'eventtype', 'label',
       'signal', 'times', 'R_peaks', 'RR_intervals_ms', 'HR_series', 'status',
       'meanHR', 'stdHR', 'minHR', 'maxHR', 'Delta_HR', 'meanRR_ms', 'SDNN_ms',
       'RMSSD_ms', 'pNN50', 'meanHRV_bpm', 'mean_ecg', 'std_ecg', 'rms_ecg',
       'ptp_ecg', 'total_power_ecg', 'bp_0_5_3_ecg', 'n_beats'],
      dtype='object')

In [159]:
feature_cols = [
    'meanHR','stdHR','minHR','maxHR','meanRR_ms','SDNN_ms','RMSSD_ms',
    'pNN50','meanHRV_bpm','mean_ecg','std_ecg','rms_ecg','ptp_ecg',
    'total_power_ecg','bp_0_5_3_ecg','n_beats'
]

X_input =ECG[feature_cols]

In [160]:
y_pred = model.predict(X_input)

In [161]:
y_proba = model.predict_proba(X_input)

In [162]:
ECG["pred_label"] = y_pred

ECG["p_class0"] = y_proba[:, 0]
ECG["p_class1"] = y_proba[:, 1]
ECG["p_class2"] = y_proba[:, 2]

In [163]:
Label_counts =ECG['pred_label'].value_counts().reset_index()
Label_counts.columns = ['Label', 'Count']
Label_counts

,Label,Count
0,1,523
1,2,129
2,0,86


In [164]:
X_fusion = np.hstack([
    ECG[["p_class0", "p_class1", "p_class2"]].values,
    ACC[["acc_score"]].values
])

y = ACC["label"].values

In [165]:
from sklearn.ensemble import RandomForestClassifier

fusion_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

fusion_model.fit(X_fusion, y)

y_pred = fusion_model.predict(X_fusion)

In [166]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y, y_pred))
print(classification_report(y, y_pred, digits=4))

[[ 86   0   0]
 [  0 540   0]
 [  0   0 112]]
              precision    recall  f1-score   support

       ictal     1.0000    1.0000    1.0000        86
      normal     1.0000    1.0000    1.0000       540
    preictal     1.0000    1.0000    1.0000       112

    accuracy                         1.0000       738
   macro avg     1.0000    1.0000    1.0000       738
weighted avg     1.0000    1.0000    1.0000       738



In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier

In [ ]:
ecg_folder = r"F:\graduation project\data\Code\final\ecg 60s 10min\features"
acc_folder = r"F:\graduation project\data\Code\final\ACC 60s 10min\features"

model_save_path = r"F:\graduation project\data\Code\final\fusion_models"
os.makedirs(model_save_path, exist_ok=True)

In [ ]:
def convert(df, label_col='label'):
    df = df.copy()
    df.loc[df[label_col] == 'postictal', label_col] = 'normal'
    return df

In [ ]:
def process_patient(patient_id):

    ecg_path = os.path.join(ecg_folder, f"ECG{patient_id}.csv")
    acc_path = os.path.join(acc_folder, f"ACC{patient_id}.csv")

    if not os.path.exists(ecg_path) or not os.path.exists(acc_path):
        print(f"Patient {patient_id} skipped (missing ECG or ACC)")
        return None


    ECG = pd.read_csv(ecg_path)
    ACC = pd.read_csv(acc_path)

    ECG = convert(ECG)
    ACC = convert(ACC)

    cols = ["acc_magnitude", "activity_score", "rms_mag", "std_mag"]

    for c in cols:
        ACC[c] = pd.to_numeric(ACC[c], errors="coerce")

    ACC[cols] = ACC[cols].fillna(0)

    ACC["acc_score"] = (
        ACC["acc_magnitude"] +
        ACC["activity_score"] +
        ACC["rms_mag"] +
        ACC["std_mag"]
    ) / 4

    ACC["acc_score"] = (
        ACC["acc_score"] - ACC["acc_score"].min()
    ) / (ACC["acc_score"].max() - ACC["acc_score"].min() + 1e-8)


    model_path = os.path.join(
        r"F:\graduation project\data\Code\final\ecg 60s 10min\patient_models100_csv",
        f"patient_{patient_id}.0_model.pkl"
    )

    if not os.path.exists(model_path):
        print(f"Patient {patient_id} skipped (missing ECG model)")
        return None

    model = joblib.load(model_path)

    feature_cols = [
        'meanHR','stdHR','minHR','maxHR','meanRR_ms','SDNN_ms','RMSSD_ms',
        'pNN50','meanHRV_bpm','mean_ecg','std_ecg','rms_ecg','ptp_ecg',
        'total_power_ecg','bp_0_5_3_ecg','n_beats'
    ]


    X_input = ECG[feature_cols]

    y_pred = model.predict(X_input)
    y_proba = model.predict_proba(X_input)

    ECG["p_class0"] = y_proba[:, 0]
    ECG["p_class1"] = y_proba[:, 1]
    ECG["p_class2"] = y_proba[:, 2]


    X_fusion = np.hstack([
        ECG[["p_class0", "p_class1", "p_class2"]].values,
        ACC[["acc_score"]].values
    ])

    y = ACC["label"].values


    fusion_model = RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    )

    fusion_model.fit(X_fusion, y)

    save_path = os.path.join(model_save_path, f"fusion_patient_{patient_id}.pkl")
    joblib.dump(fusion_model, save_path)

    print(f"Patient {patient_id} trained & saved")

    return fusion_model

In [ ]:
all_models = {}
skipped_patients = []

for pid in range(75, 126):

    model = process_patient(pid)

    if model is None:
        skipped_patients.append(pid)
    else:
        all_models[pid] = model

print("\n=========================")
print("Skipped patients:", skipped_patients)
print("Trained patients:", len(all_models))

✅ Patient 75 trained & saved
✅ Patient 76 trained & saved
❌ Patient 77 skipped (missing ECG model)
❌ Patient 78 skipped (missing ECG or ACC)
❌ Patient 79 skipped (missing ECG or ACC)
❌ Patient 80 skipped (missing ECG or ACC)
❌ Patient 81 skipped (missing ECG or ACC)
❌ Patient 82 skipped (missing ECG or ACC)
✅ Patient 83 trained & saved
✅ Patient 84 trained & saved
✅ Patient 85 trained & saved
✅ Patient 86 trained & saved
❌ Patient 87 skipped (missing ECG or ACC)
✅ Patient 88 trained & saved
✅ Patient 89 trained & saved
❌ Patient 90 skipped (missing ECG or ACC)
❌ Patient 91 skipped (missing ECG or ACC)
❌ Patient 92 skipped (missing ECG or ACC)
✅ Patient 93 trained & saved
✅ Patient 94 trained & saved
✅ Patient 95 trained & saved
✅ Patient 96 trained & saved
❌ Patient 97 skipped (missing ECG or ACC)
❌ Patient 98 skipped (missing ECG or ACC)
❌ Patient 99 skipped (missing ECG or ACC)
✅ Patient 100 trained & saved
❌ Patient 101 skipped (missing ECG or ACC)
❌ Patient 102 skipped (missing ECG

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score, recall_score

In [ ]:

results = []

for pid in range(75, 126):

    ecg_path = os.path.join(ecg_folder, f"ECG{pid}.csv")
    acc_path = os.path.join(acc_folder, f"ACC{pid}.csv")
    model_path = os.path.join(model_save_path, f"fusion_patient_{pid}.pkl")

    if (not os.path.exists(ecg_path) or
        not os.path.exists(acc_path) or
        not os.path.exists(model_path)):
        continue

    ECG = pd.read_csv(ecg_path)
    ACC = pd.read_csv(acc_path)

    ECG = convert(ECG)
    ACC = convert(ACC)

    
    cols = ["acc_magnitude", "activity_score", "rms_mag", "std_mag"]

    for c in cols:
        ACC[c] = pd.to_numeric(ACC[c], errors="coerce")

    ACC[cols] = ACC[cols].fillna(0)

    ACC["acc_score"] = (
        ACC["acc_magnitude"] +
        ACC["activity_score"] +
        ACC["rms_mag"] +
        ACC["std_mag"]
    ) / 4

    ACC["acc_score"] = (
        ACC["acc_score"] - ACC["acc_score"].min()
    ) / (ACC["acc_score"].max() - ACC["acc_score"].min() + 1e-8)


    ecg_model_path = os.path.join(
        r"F:\graduation project\data\Code\final\ecg 60s 10min\patient_models100_csv",
        f"patient_{pid}.0_model.pkl"
    )

    if not os.path.exists(ecg_model_path):
        continue

    model_ecg = joblib.load(ecg_model_path)

    feature_cols = [
        'meanHR','stdHR','minHR','maxHR','meanRR_ms','SDNN_ms','RMSSD_ms',
        'pNN50','meanHRV_bpm','mean_ecg','std_ecg','rms_ecg','ptp_ecg',
        'total_power_ecg','bp_0_5_3_ecg','n_beats'
    ]

    X_input = ECG[feature_cols]

    y_pred_ecg = model_ecg.predict(X_input)
    y_proba = model_ecg.predict_proba(X_input)

    ECG["p_class0"] = y_proba[:, 0]
    ECG["p_class1"] = y_proba[:, 1]
    ECG["p_class2"] = y_proba[:, 2]


    X_fusion = np.hstack([
        ECG[["p_class0", "p_class1", "p_class2"]].values,
        ACC[["acc_score"]].values
    ])

    y_true = ACC["label"].values


    fusion_model = joblib.load(model_path)

    y_pred = fusion_model.predict(X_fusion)


    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    rec = recall_score(y_true, y_pred, average="macro")

    results.append([pid, acc, f1, rec])


df_results = pd.DataFrame(results, columns=[
    "Patient", "Accuracy", "F1_macro", "Recall_macro"
])

print(df_results)


print("\n====================")
print("AVERAGE RESULTS")
print("====================")

print("Mean Accuracy   :", df_results["Accuracy"].mean())
print("Mean F1-score   :", df_results["F1_macro"].mean())
print("Mean Recall     :", df_results["Recall_macro"].mean())

    Patient  Accuracy  F1_macro  Recall_macro
0        75   1.00000  1.000000       1.00000
1        76   1.00000  1.000000       1.00000
2        83   1.00000  1.000000       1.00000
3        84   1.00000  1.000000       1.00000
4        85   1.00000  1.000000       1.00000
5        86   1.00000  1.000000       1.00000
6        88   1.00000  1.000000       1.00000
7        89   1.00000  1.000000       1.00000
8        93   1.00000  1.000000       1.00000
9        94   1.00000  1.000000       1.00000
10       95   1.00000  1.000000       1.00000
11       96   1.00000  1.000000       1.00000
12      100   1.00000  1.000000       1.00000
13      110   0.97705  0.773816       0.99217
14      118   1.00000  1.000000       1.00000
15      122   1.00000  1.000000       1.00000
16      123   1.00000  1.000000       1.00000
17      124   1.00000  1.000000       1.00000
18      125   1.00000  1.000000       1.00000

AVERAGE RESULTS
Mean Accuracy   : 0.9987921203616533
Mean F1-score   : 0.988095